# 使用 Deep Agents 内置 Skill 支持

运行前请在项目根目录执行 `uv sync`，在 `.env` 中配置 `DEEPSEEK_API_KEY`，然后选择项目 `.venv` 对应的 Python 内核。

请以本节 `notebooks` 目录作为工作目录运行，代码中的文件路径均相对于该目录。示例使用同目录旁的 `../skills/weekly-report/`，代码不会向周报中补造事实。初始化模型不发送请求，最后一个代码单元才调用模型并可能产生 API 费用。


In [1]:
# 1.使用相对路径加载技能和环境变量
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage


# 以本节 notebooks 目录作为工作目录运行。
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

SKILLS_DIR = Path("../skills")

# 2.沿用前文的 DeepSeek 模型；超时和重试次数用于限制演示等待时间。
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 3.使用 Deep Agents 内置的技能发现和文件读取能力
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from deepagents.middleware.filesystem import FilesystemPermission

# 虚拟路径 / 对应本节目录，不使用 Windows 绝对路径作为 skills 参数。
backend = FilesystemBackend(root_dir="..", virtual_mode=True)
agent = create_deep_agent(
    model=model,
    backend=backend,
    skills=["/skills/"],
    # 演示只读技能，最终周报直接回复；通过文件工具禁止写入本节文件。
    permissions=[FilesystemPermission(operations=["write"], paths=["/**"], mode="deny")],
    system_prompt=(
        "你是一个中文工作助手。任务匹配技能时先读取对应 SKILL.md，再读取说明要求的参考资料。"
        "仅依据用户事实回答，直接在回复中输出周报。请自行完成，不调用 task 委派任务。"
    ),
)

# 6.使用相同的虚构记录调用 Agent，打印完整消息观察加载过程
work_notes = """请把下面的工作记录整理为本周周报。
- 本周完成登录接口联调，修复了 2 个参数校验问题。
- 完成 MCP 调用笔记的初稿。
- Skill 调用示例仍在编写，尚未完成。
- 测试环境偶发超时，正在等待运维排查，问题尚未解决。
- 下周计划完成 Skill 示例，并补充登录接口的异常场景测试。
"""
response = agent.invoke(
    {"messages": [HumanMessage(work_notes)]}
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

请把下面的工作记录整理为本周周报。
- 本周完成登录接口联调，修复了 2 个参数校验问题。
- 完成 MCP 调用笔记的初稿。
- Skill 调用示例仍在编写，尚未完成。
- 测试环境偶发超时，正在等待运维排查，问题尚未解决。
- 下周计划完成 Skill 示例，并补充登录接口的异常场景测试。

================================== Ai Message ==================================

我先读取周报技能说明。
Tool Calls:
  read_file (call_00_XG2z0xCDTRNWcDnvBZCs9767)
 Call ID: call_00_XG2z0xCDTRNWcDnvBZCs9767
  Args:
    file_path: /skills/weekly-report/SKILL.md
    limit: 1000
================================= Tool Message =================================
Name: read_file

@@ lines 1-23 of 23 @@
---
name: weekly-report
description: 将零散工作记录整理为中文周报。当用户要求写周报、汇总本周工作或整理下周计划时使用。
---

# 周报整理

## 执行步骤

1. 阅读用户提供的工作记录，区分已完成、进行中、问题风险和下周计划。
2. 生成周报前，先读取本技能目录内的 `references/report-format.md`，遵循其中的格式和归类规则。
3. 按模板整理周报，合并重复事项，保留输入中的数量、状态和明确依赖。
4. 只使用用户提供的事实，不编造完成情况、指标、负责人或截止日期。
5. 缺少的信息标记为“待补充”；进行中的事项不能写成已经完成。
6. 直接在回复中输出 Markdown 周报，无需写入文件，也无需调用其他智能体。

## 读取参考文件

- 手动实现：调用 `read_skill